<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Parse many-scenarios runs and compute summary/uncertainty tables.
- **Primary inputs**: runs/many_scenarios/<run_id>/*/output.csv
- **Primary outputs**: summary.csv and scenario statistics
- **Refactor role**: Canonical parser for batch summary and uncertainty-style statistics.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Read all scenario `output.csv` files from one run folder under `scenario_analysis/runs/many_scenarios/`.
2. Compute end-year metrics and annualized investment/subsidy metrics.
3. Build run-level summary statistics, including extrema scenarios.

### Inputs
- `runs/many_scenarios/<run_id>/*/output.csv`

### Outputs
- `runs/many_scenarios/<run_id>/summary.csv`
- In-memory summary table printed in notebook.

### How To Reuse
- Change the `folder` variable in the first code cell to another run id under `runs/many_scenarios/`.


In [ ]:
import os
import pandas as pd

from project.write_output import plot_compare_scenarios_simple

In [ ]:
# Refactor: default run path under scenario_analysis/runs/many_scenarios
folder = os.path.join('runs', 'many_scenarios', 'policies_scenarios_20240528_203907')
reference = 'Reference'

In [ ]:
# Read all scenarios output from a run
ignore = ['root_log.log', 'img', '.DS_Store', 'comparison.csv', 'consumption.png', 'summary.csv']
data = dict()
for result_dir in [i for i in os.listdir(folder) if i not in ignore]:
    df = pd.read_csv(os.path.join(folder, result_dir, 'output.csv'), index_col=[0])
    start = int(df.columns[0])
    end = df.columns[-1]
    df.loc['Investment heater (Billion euro/year)', end] = df.loc['Investment heater (Billion euro)', :].sum() / (int(end) - start)
    df.loc['Investment insulation (Billion euro/year)', end] = df.loc['Investment insulation (Billion euro)', :].sum() / (int(end) - start)
    df.loc['Investment (Billion euro/year)', end] = df.loc['Investment total (Billion euro)', :].sum() / (int(end) - start)
    df.loc['Subsidies (Billion euro/year)', end] = df.loc['Subsidies total (Billion euro)', :].sum() / (int(end) - start)

    data.update({result_dir: df})
data = pd.concat((data), axis=0).rename_axis(['Scenarios', 'Variables'], axis=0)

end = data.columns[-1]

In [ ]:
variables = ['Consumption (TWh)',
             'Emission (MtCO2)',
             'Energy poverty (Million)',
             'Investment (Billion euro/year)',
             'Subsidies (Billion euro/year)',
             'Investment insulation (Billion euro/year)',
             'Investment heater (Billion euro/year)'
             ]
extract = data[data.index.get_level_values('Variables').isin(variables)].loc[:, end].unstack('Variables')
extract = extract.loc[:, variables]
extract.sort_index().to_csv(os.path.join(folder, 'summary.csv'))
reference_data = extract[extract.index.get_level_values('Scenarios') == reference]
reference_data = reference_data.rename(index={reference: 'reference'})


# select name of scenarios that equal to max and min for each column

result = pd.concat((extract.describe(), reference_data, extract.idxmax().rename('max_scenario').to_frame().T, extract.idxmin().rename('min_scenario').to_frame().T), axis=0)
result = result.loc[:, variables]
print(result)

#### Make uncertainty plots

In [ ]:
if False:
    from project.utils import make_uncertainty_plot
    variables = ['Consumption (TWh)',
                 'Emission (MtCO2)',
                 'Energy poverty (Million)']
    var = 'Emission (MtCO2)'
    var = 'Consumption (TWh)'
    test = data[data.index.get_level_values('Variables') == var].droplevel('Variables')
    make_uncertainty_plot(test.T, var, save=os.path.join(folder, 'consumption.png'), reference=reference, format_y=lambda x, _: '{:.0f} TWh'.format(x))
